## 📖 [The RAG Techniques Book is HERE](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=evaluation--rag-vs-native-file-reading-benchmark&click=book-buy-amazon&target=https%3A%2F%2Famzn.to%2F4cvxqSw&text=)

**The super extended version of this repository.** The book goes far beyond the notebooks: the **intuition** behind every technique, **side-by-side comparisons** showing when each approach wins (and when it quietly fails), and **illustrations** that make the tricky parts finally click.

⏳ **Launch window only: $0.99.** The price goes up once the launch ends, and readers who grab it now lock in the lowest price it will ever have.

### 👉 [Get the book on Amazon before the price changes](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=evaluation--rag-vs-native-file-reading-benchmark&click=book-buy-amazon&target=https%3A%2F%2Famzn.to%2F4cvxqSw&text=)

---


# RAG vs Native LLM File Reading: An Empirical Benchmark

## Overview

As LLM context windows grow larger and agentic coding tools like Claude Code gain the ability to read files directly, a natural question arises: **do you still need RAG?**

This notebook analyzes the results of an open-source benchmark that tested exactly that. Claude Code (Sonnet 4.6) was given 500 corporate PDFs and asked 10 factual questions under two configurations:

1. **Native file reading** — Claude Code searches files on its own using its built-in tools (grep, cat, read)
2. **With a RAG layer** — documents are pre-indexed, and a retrieval layer fetches relevant chunks before the LLM answers

## Motivation

The "just let the agent read the files" approach is increasingly popular. It requires zero setup — no indexing, no vector database, no chunking strategy. But does it scale?

This benchmark measures three things that matter in production:
- **Speed**: How long does each approach take as document count grows?
- **Cost**: What does each query cost in API usage?
- **Hallucination on missing information**: When the answer isn't in the documents, does the model say "I don't know" or does it fabricate an answer?

## Key Findings

| Metric | Native File Reading (500 docs) | With RAG (500 docs) | Improvement |
|--------|-------------------------------|---------------------|-------------|
| Avg response time | 2 min 31 sec | 36 sec | **4.2× faster** |
| Cost per query | $0.40 | $0.13 | **3.2× cheaper** |
| Completed in 3 min | 39% | 100% | Every query completes |
| Hallucination on missing info | 50–100% | 0% (correctly refuses) | Critical safety improvement |

## Why This Matters for RAG Practitioners

1. **RAG is not just faster — it's more honest.** The retrieval layer gives the LLM a definitive signal about what exists in the corpus before it answers. Without that signal, the model confidently fabricates.
2. **Native file reading doesn't degrade linearly.** It works fine under ~100 documents, then falls off a cliff. The non-linear scaling curve is the real finding.
3. **The "just let the agent grep" era has limits.** Agentic file-reading is a great demo. It is not yet a production system for large document collections.

## Data Source

All data in this notebook comes from the open-source benchmark at:
[github.com/adorosario/customgpt-rag-plugin-benchmarking](https://github.com/adorosario/customgpt-rag-plugin-benchmarking)

The benchmark was conducted by [CustomGPT.ai](https://customgpt.ai). As the vendor of the RAG layer tested, we acknowledge the inherent bias — the raw data, scoring logic, and reproduction scripts are fully open so anyone can verify or replicate the results with a different RAG stack.

---

# Setup and Data Loading

In [ ]:
!pip install pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

### Load the benchmark data

The summary data covers 7 document tiers (5, 10, 30, 50, 100, 250, 500) for native Claude Code, plus the 500-document RAG configuration. Each tier ran 10 questions × 3 repetitions.

In [ ]:
# Benchmark results from the open-source study
# Source: https://github.com/adorosario/customgpt-rag-plugin-benchmarking

data = {
    'Documents': [5, 10, 30, 50, 100, 250, 500],
    'Avg Wait (sec)': [35, 57, 71, 83, 113, 121, 151],
    'Cost per Query ($)': [0.11, 0.20, 0.34, 0.39, 0.36, 0.37, 0.40],
    'Completion Rate (%)': [100, 97, 97, 97, 47, 43, 39],
}

df_native = pd.DataFrame(data)

# RAG results at 500 documents
rag_results = {
    'Avg Wait (sec)': 36,
    'Cost per Query ($)': 0.13,
    'Completion Rate (%)': 100,
}

print("Native Claude Code — Scaling by Document Count")
print("=" * 55)
print(df_native.to_string(index=False))
print(f"\nWith RAG at 500 docs: {rag_results['Avg Wait (sec)']}s, ${rag_results['Cost per Query ($)']}/query, {rag_results['Completion Rate (%)']}% completion")

### Important note on right-censoring

Searches that exceeded the 3-minute (180s) benchmark window were recorded at 180 seconds, not their actual duration. This means the reported averages at 100+ documents **understate** the true wait time. The actual averages are higher.

# Visualizing the Scaling Cliff

### Completion rate drops non-linearly

Native file reading works well up to ~50 documents. Between 50 and 100, completion rate drops from 97% to 47%. This is not a gradual degradation — it's a cliff.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_native['Documents'], df_native['Completion Rate (%)'],
        marker='o', linewidth=2.5, color='#e74c3c', label='Native Claude Code', zorder=3)
ax.axhline(y=100, color='#27ae60', linestyle='--', linewidth=2, label='With RAG (500 docs)', zorder=2)

ax.fill_between(df_native['Documents'], df_native['Completion Rate (%)'], 100,
                alpha=0.1, color='#e74c3c')

ax.set_xlabel('Number of Documents', fontsize=12)
ax.set_ylabel('Queries Completed in 3 min (%)', fontsize=12)
ax.set_title('The Scaling Cliff: Native File Reading vs RAG', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 110)
ax.set_xscale('log')
ax.set_xticks(df_native['Documents'])
ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
ax.grid(True, alpha=0.3)

# Annotate the cliff
ax.annotate('Cliff: 97% \u2192 47%',
            xy=(100, 47), xytext=(150, 70),
            arrowprops=dict(arrowstyle='->', color='#e74c3c'),
            fontsize=11, color='#e74c3c', fontweight='bold')

plt.tight_layout()
plt.show()

### Response time: native vs RAG at 500 documents

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Native scaling curve
ax.plot(df_native['Documents'], df_native['Avg Wait (sec)'],
        marker='o', linewidth=2.5, color='#e74c3c', label='Native Claude Code', zorder=3)

# RAG point at 500 docs
ax.scatter([500], [rag_results['Avg Wait (sec)']],
           s=150, color='#27ae60', zorder=4, label='With RAG (500 docs)', marker='D')

# 3-minute timeout line
ax.axhline(y=180, color='gray', linestyle=':', linewidth=1.5, label='3-min timeout')

ax.set_xlabel('Number of Documents', fontsize=12)
ax.set_ylabel('Avg Response Time (seconds)', fontsize=12)
ax.set_title('Response Time Scaling: Native File Reading vs RAG', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xscale('log')
ax.set_xticks(df_native['Documents'])
ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
ax.grid(True, alpha=0.3)

# Annotate the gap
ax.annotate(f'4.2\u00d7 faster',
            xy=(500, 36), xytext=(250, 50),
            arrowprops=dict(arrowstyle='->', color='#27ae60'),
            fontsize=11, color='#27ae60', fontweight='bold')

plt.tight_layout()
plt.show()

### Cost per query comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(['Native\n(500 docs)'], [0.40], color='#e74c3c', width=0.4, label='Native Claude Code')
ax.bar(['With RAG\n(500 docs)'], [0.13], color='#27ae60', width=0.4, label='With RAG')

ax.set_ylabel('Cost per Query ($)', fontsize=12)
ax.set_title('Cost Comparison at 500 Documents', fontsize=14, fontweight='bold')

ax.bar_label(ax.containers[0], fmt='$%.2f', fontsize=13, fontweight='bold', padding=5)
ax.bar_label(ax.containers[1], fmt='$%.2f', fontsize=13, fontweight='bold', padding=5)

ax.set_ylim(0, 0.55)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# The Hallucination Problem

Speed and cost are expected advantages for retrieval. The surprising finding is about **what happens when the answer isn't in the documents**.

The benchmark includes "absent needle" questions — questions whose answers do not exist in the document set at certain tiers. This tests a critical real-world scenario: a user asks about something that isn't in their knowledge base.

In [ ]:
# Hallucination behavior on absent-needle questions
hallucination_data = {
    'Configuration': ['Native Claude Code', 'With RAG'],
    'Fabricates Answer (%)': [75, 0],  # midpoint of 50-100% range for native
    'Correctly Refuses (%)': [25, 100],
}

fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(hallucination_data['Configuration']))
width = 0.35

bars1 = ax.bar(x - width/2, hallucination_data['Fabricates Answer (%)'],
               width, label='Fabricates answer', color='#e74c3c')
bars2 = ax.bar(x + width/2, hallucination_data['Correctly Refuses (%)'],
               width, label='Correctly says "not found"', color='#27ae60')

ax.set_ylabel('Percentage of Missing-Info Queries', fontsize=12)
ax.set_title('When the Answer Isn\'t in the Documents', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(hallucination_data['Configuration'], fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis='y')

ax.bar_label(bars1, fmt='%d%%', fontsize=12, fontweight='bold', padding=3)
ax.bar_label(bars2, fmt='%d%%', fontsize=12, fontweight='bold', padding=3)

plt.tight_layout()
plt.show()

print("Note: Native Claude Code hallucination rate ranges from 50-100% depending")
print("on question type. Pattern-matching questions hallucinate ~50%, specific-fact")
print("questions hallucinate closer to 100%. The chart uses the midpoint (75%).")

# The Crossover Point

Native file reading isn't always worse. At small document counts, it's competitive — no indexing overhead, simpler setup. The question is: **where does the crossover happen?**

In [ ]:
print("At each tier, native Claude Code performance:")
print("=" * 55)
for _, row in df_native.iterrows():
    docs = int(row['Documents'])
    time = row['Avg Wait (sec)']
    rate = row['Completion Rate (%)']
    status = "OK" if rate > 90 else "DEGRADED" if rate > 50 else "FAILING"
    print(f"  {docs:>4} docs: {time:>3.0f}s avg, {rate:.0f}% completion  [{status}]")

print(f"\n  With RAG at 500 docs: {rag_results['Avg Wait (sec)']}s avg, {rag_results['Completion Rate (%)']}% completion  [OK]")
print("\n" + "=" * 55)
print("Crossover zone: ~50-100 documents.")
print("Below 50 docs: native file reading is fine.")
print("Above 100 docs: you need retrieval.")

# Known Limitations

This benchmark has real limitations that you should consider when interpreting the results:

| Limitation | Detail |
|---|---|
| **Vendor benchmark** | The RAG layer tested (CustomGPT.ai) was built by the team that ran the benchmark. Replicate with your own RAG stack. |
| **Synthetic corpus** | 500 machine-generated PDFs from one fictional company. Real documents are messier. |
| **Single model** | Only Claude Sonnet 4.6 was tested. |
| **Small N** | 10 questions × 3 runs = 30 data points per configuration. |
| **No grep/ripgrep baseline** | Claude Code was not given explicit grep tools, which could improve speed. |
| **RAG indexing cost excluded** | The $0.13/query doesn't include the one-time indexing cost. |
| **Right-censored timing** | Averages at 100+ docs understate true wait times (capped at 180s). |

Full details: [github.com/adorosario/customgpt-rag-plugin-benchmarking](https://github.com/adorosario/customgpt-rag-plugin-benchmarking)

# Takeaways for RAG Practitioners

1. **Below ~50 documents**, native LLM file reading works fine. Skip the RAG setup overhead.
2. **Between 50–100 documents**, you're in the crossover zone. Benchmark your specific use case.
3. **Above 100 documents**, you need retrieval. Not for speed (though that helps) — for correctness. The hallucination behavior on missing information is the critical failure mode.
4. **The scaling curve is non-linear.** Don't extrapolate from small-corpus demos. Test at your actual production document count.
5. **RAG doesn't just make LLMs faster — it makes them honest.** The retrieval layer provides a definitive signal about what exists in the corpus, which is the difference between "I don't know" and a confident fabrication.

---

## Reproduce or Extend This Benchmark

Everything is open source under MIT license:

```
git clone https://github.com/adorosario/customgpt-rag-plugin-benchmarking
```

The repo includes: corpus generation scripts, benchmark runners, ground truth with scoring criteria, evaluation pipeline, and all raw results.

Contributions welcome — especially replications with different RAG stacks, models, or corpora.

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=evaluation--rag-vs-native-file-reading-benchmark)